# Multi-Modal Tri-Stream AVNet & AdapterNet: Cloud GPU Training Pipeline

This notebook trains the **Tri-Stream AVNet (AVNet-Mag)** and **AdapterNet-9Axis** dead-reckoning models with GPU acceleration.

> ⚠️ **IMPORTANT FOR KAGGLE USERS**:
> In the right-hand settings panel under **Notebook options**:
> 1. Set **Accelerator** to **GPU T4 x 2** or **GPU P100**.
> 2. Toggle **Internet** to **On** (required for `git clone` and `git lfs pull`).

---

### Pipeline Steps:
1. **Environment Setup**: Detect GPU (T4 / P100 / A100) and ensure Git LFS is installed.
2. **Clone Repo & Git LFS Submodule**: Clone `https://github.com/dsainvg001/avnet.git` and pull all dataset CSV files.
3. **DataLoaders**: Build paired dataset windows ($W = 20$, $2.0\text{ s}$ @ $10\text{ Hz}$) with vehicle CAN speed ground truth.
4. **AVNet Training**: Train decoupled 3-stream CNNs + BiGRU + Attention Pooling with multi-task Huber & Geodesic loss.
5. **AdapterNet Training**: Optimize dynamic process/measurement noise covariances ($Q, N$).
6. **Trajectory Evaluation**: Run Lie-group InEKF dead reckoning and compute ATE, $E_{\text{trel}}$, $E_{\text{rrel}}$.
7. **Export Checkpoints**: Save model weights to `.pkl` and `.pth` files in `/kaggle/working/` for direct download.

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi

# Ensure Git LFS is installed (standard on Kaggle, apt fallback included)
!git lfs install 2>/dev/null || (apt-get update -qq && apt-get install -y -qq git-lfs && git lfs install)
!pip install -q torch torchvision torchaudio numpy scipy pandas matplotlib tqdm

In [ ]:
# 2. Clone Repository & Initialize Submodule with Git LFS
import os, sys

# Kaggle sets working directory to /kaggle/working
if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')
    print("Running in Kaggle environment: /kaggle/working")

# Clone the repository if not already present
if not os.path.exists('avnet'):
    print("Cloning dsainvg001/avnet with submodules...")
    !git clone --recurse-submodules https://github.com/dsainvg001/avnet.git
    %cd avnet
elif os.path.exists('avnet') and not os.path.exists('avnet/dataset.py'):
    %cd avnet

# Ensure Git LFS pulls all CSV dataset files
!git lfs install
!git submodule update --init --recursive
!git -C data lfs install
!git -C data lfs pull

sys.path.append('.')
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using compute device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

In [ ]:
# 3. Verify IO-VNBD Dataset & Git LFS Files
import os, glob
data_dir = 'data'

# Fallback: If submodule clone failed or data folder is empty
if not os.path.exists(data_dir) or len(glob.glob('data/**/*.csv', recursive=True)) == 0:
    print("Data directory empty or missing. Cloning IO-VNBD directly and pulling LFS...")
    !git clone https://github.com/onyekpeu/IO-VNBD.git data
    !cd data && git lfs install && git lfs pull && cd ..

csv_files = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
print(f"Found {len(csv_files)} total CSV files in {data_dir}/")
if csv_files:
    sample_file = csv_files[0]
    size_kb = os.path.getsize(sample_file) / 1024.0
    print(f"Sample file: {sample_file} ({size_kb:.1f} KB)")
    if size_kb < 1.0:
        print("WARNING: Pointer stub detected. Executing git lfs pull...")
        !git -C data lfs pull
    else:
        print("Git LFS verified: CSV files contain full dataset.")

In [ ]:
# 4. Verify Synchronized Dataset Pairs (S-*.csv paired with V-*.csv)
from avnet.dataset import discover_paired_iovnbd_files

pairs = discover_paired_iovnbd_files(data_dir)
print(f"Total discovered synchronized pairs: {len(pairs)}")
if pairs:
    print(f"Sample S-file (Phone IMU):       {pairs[0][0]}")
    print(f"Sample V-file (Vehicle CAN/GPS): {pairs[0][1]}")

In [ ]:
# 5. Create Train / Validation / Test DataLoaders
from avnet.dataset import create_dataloaders

# Set limit_files=None to train on all 144 sequences,
# or limit_files=10 for a faster exploratory run
train_loader, val_loader, test_loader = create_dataloaders(
    root_dir=data_dir,
    window_size=20,     # 2.0 seconds at 10 Hz
    step=2,             # 90% overlap for dense training signals
    batch_size=64,      # Optimized for GPU
    train_ratio=0.8,
    val_ratio=0.1,
    limit_files=None    # Use all files
)

batch = next(iter(train_loader))
print("Sample Batch Shapes:")
print("  Accelerometer Stream:", batch['acc'].shape)   # (B, 3, 20)
print("  Gyroscope Stream:    ", batch['gyro'].shape)  # (B, 3, 20)
print("  Magnetometer Stream: ", batch['mag'].shape)   # (B, 3, 20)
print("  Target Forward Speed:", batch['target_speed'].shape) # (B, 1)
print("  Target Delta Quat:   ", batch['target_delta_q'].shape) # (B, 3)

In [ ]:
# 6. Train Multi-Modal Tri-Stream AVNet
from avnet.models.avnet import TriStreamAVNet
from avnet.train import train_tristream_avnet

model = TriStreamAVNet(window_size=20, hidden_dim=64)

# Training Configuration
EPOCHS = 15
LEARNING_RATE = 1e-3
LAMBDA_ATT = 5.0  # Weight on attitude to eliminate heading drift

trained_avnet, history = train_tristream_avnet(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    lambda_att=LAMBDA_ATT,
    checkpoint_dir='checkpoints',
    device=device
)

In [ ]:
# 7. Plot Training and Validation Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', color='#2563eb', lw=2)
if history['val_loss']:
    axes[0].plot(history['val_loss'], label='Val Loss', color='#dc2626', lw=2)
axes[0].set_title('Multi-Task Loss (Huber + Geodesic)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Speed RMSE
if history['speed_rmse']:
    axes[1].plot(history['speed_rmse'], label='Speed RMSE (m/s)', color='#16a34a', lw=2)
    axes[1].set_title('Validation Speed RMSE (m/s)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('m/s')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

# Attitude Error
if history['att_error_deg']:
    axes[2].plot(history['att_error_deg'], label='Attitude Error (deg)', color='#9333ea', lw=2)
    axes[2].set_title('Validation Attitude Geodesic Error (°)')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Degrees')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('checkpoints/training_curves.png', dpi=300)
plt.show()

In [ ]:
# 8. Train AdapterNet-9Axis (Offline Indirect Trajectory Optimization)
from avnet.models.avnet import AdapterNet9Axis
from avnet.train_adapter import train_adapter_offline
from avnet.dataset import load_iovnbd_csv

print("Loading sequences for AdapterNet offline optimization...")
sample_sequences = [load_iovnbd_csv(s, v) for s, v in pairs[:5] if v is not None]

adapter_model = AdapterNet9Axis(in_channels=9)
trained_adapter = train_adapter_offline(
    adapter_model=adapter_model,
    avnet_model=trained_avnet,
    data_sequences=sample_sequences,
    epochs=5,
    checkpoint_dir='checkpoints',
    device=device
)

In [ ]:
# 9. Closed-Loop InEKF Trajectory Benchmark
from main import run_evaluation

test_eval_data = load_iovnbd_csv(pairs[0][0], pairs[0][1])
results = run_evaluation(
    avnet_model=trained_avnet,
    adapter_model=trained_adapter,
    eval_data=test_eval_data,
    device=device,
    output_dir='results',
    max_eval_steps=10000 # Evaluate first 10,000 steps (~16.6 minutes)
)

# Plot Estimated vs Ground Truth Path
pred_p = results['pred_positions']
gt_p = test_eval_data['gt_enu'][:len(pred_p)]

plt.figure(figsize=(10, 8))
plt.plot(gt_p[:, 0], gt_p[:, 1], 'r--', label='Ground Truth (Vehicle CAN/GPS)', lw=2.5)
plt.plot(pred_p[:, 0], pred_p[:, 1], 'b-', label='DMDVDR InEKF Dead Reckoning', lw=2.0)
plt.scatter(gt_p[0, 0], gt_p[0, 1], c='green', s=100, zorder=5, label='Start Point')
plt.title(f"Trajectory Dead Reckoning: Etrel={results['rel_metrics']['E_trel_percent']:.2f}% | ATE={results['ate']:.2f}m")
plt.xlabel('East (meters)')
plt.ylabel('North (meters)')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.savefig('results/trajectory_benchmark.png', dpi=300)
plt.show()

In [ ]:
# 10. Package & Save Trained Model to .pkl and .pth Files
import pickle

# In Kaggle, files saved under /kaggle/working/ are preserved in the Output section
model_export_pkl = 'checkpoints/avnet_tristream_model.pkl'
export_package = {
    'model_name': 'TriStreamAVNet-Mag',
    'window_size': 20,
    'hidden_dim': 64,
    'state_dict': trained_avnet.state_dict(),
    'adapter_state_dict': trained_adapter.state_dict(),
    'final_metrics': {
        'val_loss': history['val_loss'][-1] if history['val_loss'] else None,
        'speed_rmse': history['speed_rmse'][-1] if history['speed_rmse'] else None,
        'att_deg': history['att_error_deg'][-1] if history['att_error_deg'] else None
    }
}

with open(model_export_pkl, 'wb') as f:
    pickle.dump(export_package, f)

print(f"Successfully exported model package to: {model_export_pkl}")
print(f"File size: {os.path.getsize(model_export_pkl) / 1024:.2f} KB")

In [ ]:
# 11. Download / Access Model Artifacts
# In Kaggle: Check the right sidebar under 'Output' (/kaggle/working) to download directly,
# or click the clickable links below:

from IPython.display import FileLink, display

print("Generated files ready for download:")
for fpath in ['checkpoints/avnet_tristream_model.pkl', 'checkpoints/best_avnet_tristream.pth', 'results/calibration_profile.json']:
    if os.path.exists(fpath):
        print(f"  - {fpath} ({os.path.getsize(fpath) / 1024:.1f} KB)")
        display(FileLink(fpath))

# If running in Google Colab (fallback):
try:
    from google.colab import files
    files.download('checkpoints/avnet_tristream_model.pkl')
    files.download('checkpoints/best_avnet_tristream.pth')
    files.download('results/calibration_profile.json')
except ImportError:
    pass